In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    raise RuntimeError("CUDA is not available!")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.0 GB


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully!")

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in I:\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1123 20:29:09.986000 18544 Lib\site-packages\torch\dis

Model loaded successfully!


In [4]:
def ask_model(question):

    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return response.strip()

In [5]:
questions = {
    "English": "What is the capital of Bangladesh? Answer in English.",

    "Bangla": "বাংলাদেশের রাজধানীর নাম কী? বাংলায় উত্তর দাও।",

    "Banglish": "Bangladesh er rajdhani ki? Banglish e uttor dao?"
}

for language, question in questions.items():

    print(f"\n{'='*50}")
    print(language)
    print("="*50)

    answer = ask_model(question)

    print("Question:", question)
    print("Answer:", answer)


English
Question: What is the capital of Bangladesh? Answer in English.
Answer: The capital of Bangladesh is Dhaka.

Bangla
Question: বাংলাদেশের রাজধানীর নাম কী? বাংলায় উত্তর দাও।
Answer: বাংলাদেশের রাজধানীর নাম Dhaka এবং সংবাদপত্রের ভাষা বাংলা হওয়ায় এটি বাংলায় উত্তর দিতে পারি:

Dhaka

Banglish
Question: Bangladesh er rajdhani ki? Banglish e uttor dao?
Answer: Bangladesh ke rajdhani Bangladesh taka (Dhaka) holo. Dhaka er bhasha bangla e "বাংলাদেশ" tahole itni jante hobe.

Unke ar uttor (up) bhasha Bangla e "পূর্ব" holo. Bangladesh er uttor bhalobasha (state) "Chittagong" holo. 

Dhaka er uttor bhalob


In [6]:
print(
    "GPU Memory Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU Memory Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU Memory Allocated: 5.76 GB
GPU Memory Reserved: 5.83 GB


In [8]:
import json
from pathlib import Path

# Load questions
with open("data/questions_english.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

results = []

# Generate answers using Qwen
for i, item in enumerate(questions, start=1):

    question = item["question"]

    print(f"\nQuestion {i}/{len(questions)}")
    print("Q:", question)

    answer = ask_model(question)

    print("A:", answer)

    results.append({
        "question": question,
        "answer": answer
    })

output_path = Path(
    r"I:\Capstone\knowledge-conflict-pilot\outputs\english_base_answer.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Saved to: {output_path}")


Question 1/50
Q: What is the capital of Australia?
A: The capital of Australia is Canberra.

Question 2/50
Q: Which is the largest country in the world by area?
A: The largest country in the world by land area is Russia.

Question 3/50
Q: Who was the first person to walk on the Moon?
A: The first person to walk on the Moon was Neil Armstrong. On July 20, 1969, as part of the Apollo 11 mission, Armstrong made his famous statement "That's one small step for man, one giant leap for mankind" as he stepped down from the lunar module Eagle and onto the Moon's surface. He was accompanied by fellow astronaut Edwin "Buzz" Aldrin Jr., who also walked on the Moon. The third member of the crew, Michael Collins

Question 4/50
Q: What is the national animal of Bangladesh?
A: The national animal of Bangladesh is the Bengal tiger (Panthera tigris tigris).

Question 5/50
Q: Which planet is closest to the Sun?
A: The planet closest to the Sun is Mercury.

Question 6/50
Q: Who wrote the national anthem 

In [9]:
import json
from pathlib import Path

# Load questions
with open("data/questions_bangla.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

results = []

# Generate answers using Qwen
for i, item in enumerate(questions, start=1):

    question = item["question"]

    print(f"\nQuestion {i}/{len(questions)}")
    print("Q:", question)

    answer = ask_model(question)

    print("A:", answer)

    results.append({
        "question": question,
        "answer": answer
    })

output_path = Path(
    r"I:\Capstone\knowledge-conflict-pilot\outputs\bangla_base_answer.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Saved to: {output_path}")


Question 1/50
Q: অস্ট্রেলিয়ার রাজধানীর নাম কী?
A: অস্ট্রেলিয়ার রাজধানীর নাম Sydney (স্যাডিয়া) হয়।

Question 2/50
Q: আয়তনের দিক থেকে পৃথিবীর সবচেয়ে বড় দেশ কোনটি?
A: পৃথিবীর আয়তনের দিক থেকে সবচেয়ে বড় দেশ হল জাপান। এটি প্রায় 377,975 কিলোমিটার বর্তমান আয়ত

Question 3/50
Q: চাঁদে প্রথম পা রাখা ব্যক্তি কে ছিলেন?
A: চাঁদে প্রথম পা রাখা ব্যক্তি হচ্ছে আলিউমিনিয়াম। এটি মানুষের জন্য একটি সম্ভাব্য ব্যাপার নয়, কারণ �

Question 4/50
Q: বাংলাদেশের জাতীয় প্রাণী কোনটি?
A: বাংলাদেশের জাতীয় প্রাণী হল বাঙালি চিকুতি। এটি একটি স্বাভাবিক প্রাণী যা বাংলাদেশের প্রাকৃতিক প

Question 5/50
Q: সূর্যের সবচেয়ে কাছের গ্রহ কোনটি?
A: সূর্যের সবচেয়ে কাছের গ্রহ হল জোর্ডান। এটি আমেরিকান পদার্থবিদ জোসেফ জোর্ডান নামক একজন বিখ্যাত শ

Question 6/50
Q: বাংলাদেশের জাতীয় সংগীত কে রচনা করেছেন?
A: বাংলাদেশের জাতীয় সংগীত কে রচনা করেছেন নিউজিল্যান্ডে বাসী একজন বাংলা ভাষার শিল্পী ও সংগীতশিল

Question 7/50
Q: পৃথিবীর দীর্ঘতম নদী কোনটি?
A: পৃথিবীর দীর্ঘতম নদী হল জাম্পিং (Jamuna)। এটি ভারত-বাংলাদেশ সংযোগের একটি প্রধ

In [10]:
import json
from pathlib import Path

# Load questions
with open("data/questions_banglish.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

results = []

# Generate answers using Qwen
for i, item in enumerate(questions, start=1):

    question = item["question"]

    print(f"\nQuestion {i}/{len(questions)}")
    print("Q:", question)

    answer = ask_model(question)

    print("A:", answer)

    results.append({
        "question": question,
        "answer": answer
    })

output_path = Path(
    r"I:\Capstone\knowledge-conflict-pilot\outputs\banglish_base_answer.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Saved to: {output_path}")


Question 1/50
Q: Australia-r rajdhanir nam ki?
A: I'm sorry, but the question "Australia-r rajdhanir nam ki?" doesn't have a clear meaning in English. It appears to be a mix of words or a typo. Could you please clarify your question or provide more context? Are you asking about someone named Rajdhan in Australia? Or is there another language or context I should consider?

Question 2/50
Q: Ayotoner dik theke prithibir shobcheye boro desh konti?
A: I'm sorry, but your question is not clear in Bengali. It seems like you might be asking about which country has the largest number of protected areas or biodiversity hotspots. Could you please clarify your question or provide more context? I'd be happy to help if you can rephrase it.

Question 3/50
Q: Chande prothom pa rakha bekti ke chilen?
A: I'm sorry, but the question "Chande prothom pa rakha bekti ke chilen?" doesn't appear to have a clear meaning in English or Bengali that I can understand. It seems like it might be misspelled or not pr